In [1]:
import pandas as pd
import numpy as np

def process_buy_hold_trades(file_path: str):
    """
    End-to-end Buy-Hold strategy PnL computation.

    Logic
    -----
    1. Identify trades using:
       - Month gaps
       - Quantity changes
    2. Compute per-trade returns
    3. Compute compounded strategy return

    Returns
    -------
    bh_pivot : pd.DataFrame
        Trade-level details with buy, sell, PnL, dates
    strategy_return : float
        Compounded strategy return
    """

    # ----------------------------
    # Step 1: Read & prepare data
    # ----------------------------
    df = pd.read_excel(file_path)

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Ticker", "Date"])

    df["YearMonth"] = df["Date"].dt.to_period("M")

    df = df[["Ticker", "Date", "YearMonth", "Buy_Hold_Value", "Quantity"]]

    # ----------------------------
    # Step 2: Monthly snapshot
    # ----------------------------
    df_month = (
        df.drop_duplicates(subset=["Ticker", "YearMonth"])
          .sort_values(["Ticker", "YearMonth"])
    )

    # ----------------------------
    # Step 3: Trade ID logic
    # ----------------------------
    def get_trade_id(g):
        diff_months = g["YearMonth"].diff().apply(
            lambda x: x.n if pd.notna(x) else 1
        )

        new_trade = (
            diff_months.isna() |                # first row
            (diff_months > 1) |                 # month gap
            (g["Quantity"] != g["Quantity"].shift(1))  # quantity change
        )

        return new_trade.cumsum()

    df_month["trade_id"] = (
        df_month.groupby("Ticker", group_keys=False)
                .apply(get_trade_id)
    )

    # ----------------------------
    # Step 4: Merge trade_id back
    # ----------------------------
    df = df.merge(
        df_month[["Ticker", "YearMonth", "trade_id"]],
        on=["Ticker", "YearMonth"],
        how="left"
    )

    # ----------------------------
    # Step 5: Identify buy / sell
    # ----------------------------
    first_date = df.groupby(["Ticker", "trade_id"])["Date"].transform("min")
    last_date  = df.groupby(["Ticker", "trade_id"])["Date"].transform("max")

    df["Transaction"] = np.select(
        [
            df["Date"] == first_date,
            df["Date"] == last_date
        ],
        ["buy", "sell"],
        default=""
    )

    # ----------------------------
    # Step 6: Pivot to trade level
    # ----------------------------
    trades = df[df["Transaction"] != ""]

    bh_pivot = (
        trades.pivot_table(
            index=["Ticker", "trade_id"],
            columns="Transaction",
            values="Buy_Hold_Value",
            aggfunc="first"
        )
        .reset_index()
    )

    # ----------------------------
    # Step 7: Per-trade return
    # ----------------------------
    bh_pivot["Profit/Loss%"] = bh_pivot["sell"] / bh_pivot["buy"] - 1
    bh_pivot["Profit/Loss"] = bh_pivot["sell"] - bh_pivot["buy"]

    # ----------------------------
    # Step 8: Add Buy/Sell dates
    # ----------------------------
    buy_dates = trades[trades["Transaction"] == "buy"][
        ["Ticker", "trade_id", "Date"]
    ].rename(columns={"Date": "Buy Date"})

    sell_dates = trades[trades["Transaction"] == "sell"][
        ["Ticker", "trade_id", "Date"]
    ].rename(columns={"Date": "Sell Date"})

    bh_pivot = (
        bh_pivot.merge(buy_dates, on=["Ticker", "trade_id"])
                 .merge(sell_dates, on=["Ticker", "trade_id"])
    )

    # ----------------------------
    # Step 9: Strategy return (CRITICAL)
    # ----------------------------
    strategy_return = (1 + bh_pivot["Profit/Loss"]).prod() - 1

    return bh_pivot, strategy_return

#Absolute logic, if a stock remains for more than a month inside the portfolio without quantity change, the return shall be calculated for the whole period, eg 3 months

In [2]:
result = process_buy_hold_trades('Momentum_Maxfolio.xlsx')[0]
result

C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: RuntimeWarning: overflow encountered in scalar multiply
  new_data = np.array([self.freq.base * x for x in new_i8_data])
C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: RuntimeWarning: overflow encountered in scalar multiply
  new_data = np.array([self.freq.base * x for x in new_i8_data])
C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: RuntimeWarning: overflow encountered in scalar multiply
  new_data = np.array([self.freq.base * x for x in new_i8_data])
C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: RuntimeWarning: overflow encountered in scalar multiply
  new_data = np.array([self.freq.base * x for x in new_i8_data])
C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: R

,Ticker,trade_id,buy,sell,Profit/Loss%,Profit/Loss,Buy Date,Sell Date
0,ABCAPITAL,1,3.659772,3.676215,0.004493,0.016444,2025-12-01,2025-12-31
1,AIIL,1,3.750000,3.704489,-0.012136,-0.045511,2025-11-11,2025-11-28
2,ANANDRATHI,1,3.750000,3.511743,-0.063535,-0.238257,2025-11-11,2025-11-28
3,ANANDRATHI,2,3.692864,3.655636,-0.010081,-0.037228,2026-01-01,2026-01-22
4,AUBANK,1,3.661515,3.831011,0.046291,0.169497,2025-12-01,2025-12-31
5,CANBK,1,3.750000,4.035103,0.076028,0.285103,2025-11-11,2025-11-28
6,CANBK,2,3.697683,3.727889,0.008169,0.030207,2026-01-01,2026-01-22
7,CUMMINSIND,1,3.750000,3.433974,-0.084273,-0.316026,2025-11-11,2026-01-22
8,DELHIVERY,1,3.750000,3.715996,-0.009068,-0.034004,2025-11-11,2025-11-28
9,EICHERMOT,1,3.717375,3.817280,0.026875,0.099906,2025-12-01,2026-01-22


In [3]:
import numpy as np

mapping = {
    "SILVERBEES": "SILVER",
    "SILVER": "SILVER",
    "GOLDBEES": "GOLD",
    "MOGSEC": "DEBT"
}

result["Asset_Class"] = result["Ticker"].map(mapping).fillna("EQUITIES")
result


,Ticker,trade_id,buy,sell,Profit/Loss%,Profit/Loss,Buy Date,Sell Date,Asset_Class
0,ABCAPITAL,1,3.659772,3.676215,0.004493,0.016444,2025-12-01,2025-12-31,EQUITIES
1,AIIL,1,3.750000,3.704489,-0.012136,-0.045511,2025-11-11,2025-11-28,EQUITIES
2,ANANDRATHI,1,3.750000,3.511743,-0.063535,-0.238257,2025-11-11,2025-11-28,EQUITIES
3,ANANDRATHI,2,3.692864,3.655636,-0.010081,-0.037228,2026-01-01,2026-01-22,EQUITIES
4,AUBANK,1,3.661515,3.831011,0.046291,0.169497,2025-12-01,2025-12-31,EQUITIES
5,CANBK,1,3.750000,4.035103,0.076028,0.285103,2025-11-11,2025-11-28,EQUITIES
6,CANBK,2,3.697683,3.727889,0.008169,0.030207,2026-01-01,2026-01-22,EQUITIES
7,CUMMINSIND,1,3.750000,3.433974,-0.084273,-0.316026,2025-11-11,2026-01-22,EQUITIES
8,DELHIVERY,1,3.750000,3.715996,-0.009068,-0.034004,2025-11-11,2025-11-28,EQUITIES
9,EICHERMOT,1,3.717375,3.817280,0.026875,0.099906,2025-12-01,2026-01-22,EQUITIES


In [4]:
(result.groupby("Asset_Class")["Profit/Loss"].sum() / 100).reset_index()

#copy paste values from here to google sheets

,Asset_Class,Profit/Loss
0,DEBT,-0.000153
1,EQUITIES,-0.002111
2,GOLD,0.025496
3,SILVER,0.032317


In [5]:
result.to_excel('PnL_Momentum_Maxfolio.xlsx', index=False)
